# SAMS Vision — Distance Estimation Training

Train the **pose-geometry residual distance NN** used by live SAMS vision:

\[
Z = Z_0 \cdot e^{\delta},\quad Z_0 = \frac{f_y \cdot H}{h_{\mathrm{px}}},\quad \delta = f_\theta(x)
\]

## Dataset

[Human tracking dataset of 3D anatomical landmarks and pose keypoints](https://data.mendeley.com/datasets/493s6f753v/2)
(DOI: `10.17632/493s6f753v.2`, CC BY 4.0) — Instituto de Biomecánica de Valencia.

Each sample provides:
- **MediaPipe 33×2D keypoints** + camera `proj_matrix` (`*__KP2D.json`)
- **Triangulated 3D keypoints** in metres (`*__KP3D.trc`)
- Subject **height / sex / age** (`SUBJECTS_CHARACTERISTICS.csv`)

Ground-truth camera–person distance is \(\|X_{\mathrm{hip}} - C_{\mathrm{cam}}\|\) from the 3D hips and the camera centre (null-space of \(P\)).

> Ring cameras sit at nearly fixed radius (~3.6 m). For **long-range** coverage (SAMS vision band, ToF handles near field) we apply **pinhole virtual-distance augmentation**: scale 2D keypoints as if the same pose were seen at other metric ranges. Poses and anthropometry stay real; only range is resampled.

## Goals
- Download & inspect the Mendeley subset (do not train blindly)
- EDA: distance histogram, pose coverage, subject heights
- ≥ 5 experiments with different approaches
- Primary metrics: **MAE, MedAE, RMSE** (metres) + relative error
- Export weights to `models/vision/distance/distance_nn.pt` for Django live inference

## Experiment plan
| # | Approach | Model | Key idea |
|---|----------|-------|----------|
| 1 | Geometry only | Pinhole \(Z_0\) | No learning |
| 2 | Linear baseline | Ridge on features | Sanity check |
| 3 | Residual MLP | SAMS architecture | \(Z=Z_0 e^{\delta}\) |
| 4 | Absolute MLP | MLP → \(Z\) directly | No geometric prior |
| 5 | Residual + strong pose | Same MLP, pose-heavy | Drop weak bbox-only rows |
| 6 | Residual log-Huber | SAMS MLP + Huber(log Z) | Robust long-range fit |


## 0. Colab setup — directories, Drive, packages


In [ ]:
# --- Colab environment -------------------------------------------------------
import os
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)

SAMS_ROOT = Path("/content/SAMS") if IN_COLAB else Path("./SAMS_runtime")
for p in [
    SAMS_ROOT / "datasets" / "vision" / "distance" / "mendeley_493s6f753v",
    SAMS_ROOT / "models" / "vision" / "distance",
    SAMS_ROOT / "results" / "vision" / "distance",
]:
    p.mkdir(parents=True, exist_ok=True)

print("SAMS_ROOT:", SAMS_ROOT.resolve())


In [ ]:
# --- Mount Google Drive (persist trained models / results) -------------------
DRIVE_ROOT = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/SAMS")
else:
    # Local: prefer the real repo if we are inside it
    here = Path.cwd().resolve()
    repo = here if (here / "manage.py").exists() else (here.parent if (here.parent / "manage.py").exists() else Path("./SAMS_drive_mirror"))
    DRIVE_ROOT = repo

for p in [
    DRIVE_ROOT / "models" / "vision" / "distance",
    DRIVE_ROOT / "results" / "vision" / "distance",
    DRIVE_ROOT / "datasets" / "vision" / "distance",
]:
    p.mkdir(parents=True, exist_ok=True)

MODELS_DIR = DRIVE_ROOT / "models" / "vision" / "distance"
RESULTS_DIR = DRIVE_ROOT / "results" / "vision" / "distance"
DATA_DIR = SAMS_ROOT / "datasets" / "vision" / "distance" / "mendeley_493s6f753v"
print("MODELS_DIR:", MODELS_DIR)
print("DATA_DIR:", DATA_DIR)


In [ ]:
# --- Install dependencies ----------------------------------------------------
import subprocess
import sys

pkgs = ["scikit-learn", "seaborn", "tqdm", "torch", "pandas", "matplotlib", "numpy"]
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    print("Packages installed.")
else:
    print("Skipping pip install outside Colab (use project venv).")


In [ ]:
# --- Imports & reproducibility -----------------------------------------------
import gc
import json
import math
import time
import warnings
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from numpy.linalg import svd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


## 1. Download Mendeley subset (API)

Public file listing: `https://data.mendeley.com/api/datasets/493s6f753v/files`  
(DOI page: https://data.mendeley.com/datasets/493s6f753v/2)

We pull **SUBJECTS_CHARACTERISTICS.csv** plus **A-POSE** and **RUNNING** KP2D/KP3D for subjects `TDB_001`–`TDB_004` (compact, multi-view). Larger jump/gait archives are optional.


In [ ]:
# --- Mendeley file index + selective download --------------------------------
import urllib.request

MENDELEY_ID = "493s6f753v"
DOI = "10.17632/493s6f753v.2"
API_FILES = f"https://data.mendeley.com/api/datasets/{MENDELEY_ID}/files"

UA = {"User-Agent": "SAMS-distance-notebook/1.0 (research; campus project)"}

def http_json(url: str) -> Any:
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.load(resp)

def download(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=300) as resp, open(dest, "wb") as f:
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)

print("Fetching Mendeley file index (first page)…")
file_index = http_json(API_FILES)
print(f"Indexed {len(file_index)} files on page 1 (API returns ≤100; full set is larger on the website).")

by_name = {f["filename"]: f for f in file_index}

SUBJECTS = ["TDB_001_F", "TDB_002_F", "TDB_003_M", "TDB_004_F"]
MOVEMENTS = ["A-POSE", "RUNNING"]  # compact + some locomotion
want = ["SUBJECTS_CHARACTERISTICS.csv"]
for subj in SUBJECTS:
    for mov in MOVEMENTS:
        for kind, ext in [("KP2D", "json"), ("KP3D", "trc")]:
            want.append(f"{subj}__{mov}__{kind}.{ext}")

missing = [w for w in want if w not in by_name]
print("Requested:", len(want), "| missing from API page:", missing)

for name in want:
    if name not in by_name:
        continue
    meta = by_name[name]
    dest = DATA_DIR / name
    print(f"↓ {name} ({meta['size']/1e6:.2f} MB)")
    download(meta["content_details"]["download_url"], dest)

print("Done. Files on disk:", sorted(p.name for p in DATA_DIR.glob('*'))[:20], "…")


## 2. Parse KP2D / KP3D → metric distance + pose features


In [ ]:
# --- Parsers: TRC 3D, camera centre, MediaPipe features ----------------------
FEATURE_NAMES = [
    "rel_h", "rel_w", "aspect", "cx_norm", "cy_norm", "bottom_y_norm",
    "inv_rel_h", "log_z0", "det_conf", "pose_available",
    "pose_height_rel", "shoulder_width_rel", "hip_width_rel", "torso_rel",
    "pose_conf_mean", "pose_inv_height",
]

# MediaPipe Pose landmark indices
MP_NOSE, MP_L_SHO, MP_R_SHO = 0, 11, 12
MP_L_HIP, MP_R_HIP = 23, 24
MP_L_ANK, MP_R_ANK = 27, 28


def load_subjects_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={
        "Subject_code": "subject",
        "Height (cm)": "height_cm",
        "Weight (kg)": "weight_kg",
        "Age (years)": "age",
        "Sex": "sex",
    })
    df["height_m"] = df["height_cm"].astype(float) / 100.0
    return df


def load_trc_frames(path: Path) -> Dict[int, np.ndarray]:
    """Return {frame_idx: (33,3) XYZ metres}."""
    out: Dict[int, np.ndarray] = {}
    for ln in path.read_text(errors="replace").splitlines():
        if not ln or not ln[0].isdigit() or ln.startswith("90\t90"):
            continue
        parts = ln.split("\t")
        if len(parts) < 2 + 99:
            continue
        try:
            fr = int(float(parts[0]))
            xyz = np.array([float(x) for x in parts[2:2 + 99]], dtype=np.float64).reshape(33, 3)
        except ValueError:
            continue
        out[fr] = xyz
    return out


def camera_center_from_P(P: np.ndarray) -> np.ndarray:
    """Camera centre C in world coords: P @ [C;1] = 0."""
    _, _, vh = svd(P.astype(np.float64))
    C = vh[-1]
    return C[:3] / C[3]


def estimate_image_size(keypoints_uvs: List[np.ndarray]) -> Tuple[int, int]:
    umax = max(float(k[:, 0].max()) for k in keypoints_uvs) if keypoints_uvs else 480
    vmax = max(float(k[:, 1].max()) for k in keypoints_uvs) if keypoints_uvs else 480
    # MediaPipe virtual views in this set are roughly square ~414; pad to safe canvas
    W = int(max(480, math.ceil(umax / 32) * 32 + 32))
    H = int(max(480, math.ceil(vmax / 32) * 32 + 32))
    return W, H


def mp_to_features(
    kps: np.ndarray,
    image_wh: Tuple[int, int],
    *,
    person_height_m: float,
    fy_px: float,
    det_conf: float = 0.9,
) -> Tuple[np.ndarray, float, List[float]]:
    """Build SAMS FEATURE_NAMES vector + Z0 from MediaPipe (33,3)=(u,v,score).

    Also returns xyxy bbox [x1,y1,x2,y2] for optional viz / YOLO-parity.
    """
    w, h = image_wh
    conf = kps[:, 2]
    valid = conf >= 0.3
    if valid.sum() < 4:
        raise ValueError("too few keypoints")

    u = kps[:, 0]
    v = kps[:, 1]
    x1, x2 = float(u[valid].min()), float(u[valid].max())
    y1, y2 = float(v[valid].min()), float(v[valid].max())
    bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
    rel_h, rel_w = bh / h, bw / w
    aspect = bh / bw
    cx_norm = ((x1 + x2) * 0.5) / w
    cy_norm = ((y1 + y2) * 0.5) / h
    bottom_y_norm = y2 / h

    pose_available = 0.0
    pose_height_rel = shoulder_width_rel = hip_width_rel = torso_rel = 0.0
    pose_conf_mean = float(conf[valid].mean())
    pose_inv_height = 0.0
    height_px = bh

    def ok(i: int) -> bool:
        return float(conf[i]) >= 0.3

    if ok(MP_NOSE) and ok(MP_L_ANK) and ok(MP_R_ANK):
        foot_y = 0.5 * (v[MP_L_ANK] + v[MP_R_ANK])
        head_y = v[MP_NOSE]
        if foot_y > head_y:
            pose_h = float(foot_y - head_y)
            pose_height_rel = pose_h / h
            pose_inv_height = 1.0 / max(pose_height_rel, 1e-3)
            height_px = max(height_px, pose_h)
            pose_available = 1.0
    if ok(MP_L_SHO) and ok(MP_R_SHO):
        shoulder_width_rel = abs(float(u[MP_R_SHO] - u[MP_L_SHO])) / w
        pose_available = 1.0
    if ok(MP_L_HIP) and ok(MP_R_HIP):
        hip_width_rel = abs(float(u[MP_R_HIP] - u[MP_L_HIP])) / w
        pose_available = 1.0
    if ok(MP_L_SHO) and ok(MP_R_SHO) and ok(MP_L_HIP) and ok(MP_R_HIP):
        sho_y = 0.5 * (v[MP_L_SHO] + v[MP_R_SHO])
        hip_y = 0.5 * (v[MP_L_HIP] + v[MP_R_HIP])
        torso_rel = abs(float(hip_y - sho_y)) / h
        pose_available = 1.0

    z0 = (fy_px * person_height_m) / max(height_px, 1.0)
    z0 = float(np.clip(z0, 0.5, 12.0))
    feats = np.array([
        rel_h, rel_w, aspect, cx_norm, cy_norm, bottom_y_norm,
        1.0 / max(rel_h, 1e-3), math.log(max(z0, 1e-3)), float(det_conf), pose_available,
        pose_height_rel, shoulder_width_rel, hip_width_rel, torso_rel,
        pose_conf_mean, pose_inv_height,
    ], dtype=np.float32)
    return feats, z0, [x1, y1, x2, y2]


def scale_keypoints_for_distance(
    kps: np.ndarray,
    z_ref: float,
    z_new: float,
    principal: Tuple[float, float],
) -> np.ndarray:
    """Pinhole virtual-range: scale (u,v) about principal point by z_ref/z_new."""
    s = float(z_ref / max(z_new, 1e-3))
    out = kps.copy()
    cx, cy = principal
    out[:, 0] = cx + (kps[:, 0] - cx) * s
    out[:, 1] = cy + (kps[:, 1] - cy) * s
    return out


In [ ]:
# --- Build labeled table (real GT + virtual long-range augmentation) ---------
FY_PX = 720.0  # matches SAMS calib default; Z0 prior only
subjects_df = load_subjects_csv(DATA_DIR / "SUBJECTS_CHARACTERISTICS.csv")
height_map = dict(zip(subjects_df["subject"], subjects_df["height_m"]))

# Virtual distances for long-range NN (ToF owns ≤ ~1.5 m in deployment)
VIRTUAL_DISTANCES = np.array([1.8, 2.2, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0], dtype=float)
MAX_CAMERAS_PER_FRAME = 12  # subsample ring views
FRAME_STRIDE = 2

rows: List[Dict[str, Any]] = []
skipped = 0

for subj in SUBJECTS:
    H = float(height_map.get(subj, 1.70))
    for mov in MOVEMENTS:
        kp2d_path = DATA_DIR / f"{subj}__{mov}__KP2D.json"
        kp3d_path = DATA_DIR / f"{subj}__{mov}__KP3D.trc"
        if not kp2d_path.exists() or not kp3d_path.exists():
            print("skip missing", kp2d_path.name)
            continue
        meta = json.loads(kp2d_path.read_text())
        trc = load_trc_frames(kp3d_path)
        # image size from all keypoints in this sequence
        all_k = []
        for ann in meta["annotations"]:
            ks = np.asarray(ann["keypoints_scores"], dtype=float).reshape(33, 3)
            all_k.append(ks)
        img_wh = estimate_image_size(all_k)
        cx_img, cy_img = img_wh[0] / 2.0, img_wh[1] / 2.0

        # group annotations by frame
        by_frame: Dict[int, List[dict]] = {}
        for ann in meta["annotations"]:
            by_frame.setdefault(int(ann["frame"]), []).append(ann)

        frames = sorted(by_frame)
        frames = frames[::FRAME_STRIDE]
        for fr in frames:
            if fr not in trc:
                skipped += 1
                continue
            xyz = trc[fr]
            hip = 0.5 * (xyz[MP_L_HIP] + xyz[MP_R_HIP])
            anns = by_frame[fr]
            # deterministic camera subsample
            anns = sorted(anns, key=lambda a: int(a["camera"]))[:: max(1, len(anns) // MAX_CAMERAS_PER_FRAME)]
            anns = anns[:MAX_CAMERAS_PER_FRAME]
            for ann in anns:
                P = np.asarray(ann["proj_matrix"], dtype=float).reshape(3, 4)
                C = camera_center_from_P(P)
                z_ref = float(np.linalg.norm(hip - C))
                if not np.isfinite(z_ref) or z_ref < 0.5 or z_ref > 15:
                    skipped += 1
                    continue
                kps0 = np.asarray(ann["keypoints_scores"], dtype=float).reshape(33, 3)
                mean_conf = float(kps0[:, 2].mean())
                # Always keep the true measured distance
                targets = [z_ref] + [float(z) for z in VIRTUAL_DISTANCES if abs(z - z_ref) > 0.25]
                for z_tgt in targets:
                    try:
                        kps = kps0 if abs(z_tgt - z_ref) < 1e-6 else scale_keypoints_for_distance(
                            kps0, z_ref, z_tgt, (cx_img, cy_img)
                        )
                        feats, z0, bbox = mp_to_features(
                            kps, img_wh, person_height_m=H, fy_px=FY_PX, det_conf=mean_conf
                        )
                    except ValueError:
                        skipped += 1
                        continue
                    row = {
                        "subject": subj,
                        "movement": mov,
                        "frame": fr,
                        "camera": int(ann["camera"]),
                        "distance_m": float(z_tgt),
                        "z_ref_m": float(z_ref),
                        "z0_m": float(z0),
                        "augmented": abs(z_tgt - z_ref) >= 1e-6,
                        "height_m": H,
                        "img_w": img_wh[0],
                        "img_h": img_wh[1],
                    }
                    for i, name in enumerate(FEATURE_NAMES):
                        row[name] = float(feats[i])
                    rows.append(row)

df = pd.DataFrame(rows)
out_csv = DATA_DIR / "distance_features_labeled.csv"
df.to_csv(out_csv, index=False)
print(f"Rows: {len(df):,} | skipped builds: {skipped} | subjects: {df['subject'].nunique()}")
print(f"Real-only rows: {(~df['augmented']).sum():,} | augmented: {df['augmented'].sum():,}")
print(f"Distance range: {df['distance_m'].min():.2f} … {df['distance_m'].max():.2f} m")
print("Saved:", out_csv)
display(df.head())


## 3. EDA


In [ ]:
# --- EDA plots ---------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df.loc[~df["augmented"], "distance_m"], bins=30, color="#2980b9", alpha=0.85, label="measured")
axes[0, 0].hist(df.loc[df["augmented"], "distance_m"], bins=30, color="#e67e22", alpha=0.45, label="virtual range")
axes[0, 0].set_title("Camera–person distance")
axes[0, 0].set_xlabel("metres")
axes[0, 0].legend()

sns.countplot(data=df.drop_duplicates(["subject", "movement", "frame", "camera"]),
              y="movement", ax=axes[0, 1], color="#27ae60")
axes[0, 1].set_title("Views by movement (unique camera-frames)")

axes[1, 0].scatter(df["z0_m"], df["distance_m"], s=6, alpha=0.15, c=df["augmented"].map({False: "#2980b9", True: "#e67e22"}))
axes[1, 0].plot([0, 12], [0, 12], "k--", lw=1)
axes[1, 0].set_xlabel("Z0 pinhole prior (m)")
axes[1, 0].set_ylabel("GT distance (m)")
axes[1, 0].set_title("Geometry prior vs label")

subj_h = subjects_df[subjects_df["subject"].isin(SUBJECTS)]
axes[1, 1].barh(subj_h["subject"], subj_h["height_m"], color="#8e44ad")
axes[1, 1].set_xlabel("height (m)")
axes[1, 1].set_title("Subject heights (Mendeley CSV)")

plt.tight_layout()
fig_path = RESULTS_DIR / "eda_distance_mendeley.png"
plt.savefig(fig_path, dpi=140, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)
print(df[["distance_m", "z0_m", "pose_available", "pose_height_rel"]].describe().T)


## 4. Train / val split (group by subject)


In [ ]:
# --- Splits: hold out subjects to avoid leakage ------------------------------
feature_cols = FEATURE_NAMES
X = df[feature_cols].to_numpy(np.float32)
y = df["distance_m"].to_numpy(np.float32)
z0 = df["z0_m"].to_numpy(np.float32)
groups = df["subject"].to_numpy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X[train_idx], X[test_idx]
y_tr, y_te = y[train_idx], y[test_idx]
z0_tr, z0_te = z0[train_idx], z0[test_idx]

print("Train:", len(train_idx), "Test:", len(test_idx))
print("Train subjects:", sorted(set(groups[train_idx])))
print("Test subjects:", sorted(set(groups[test_idx])))


In [ ]:
# --- Metrics + experiment bookkeeping ----------------------------------------
def regression_metrics(y_true, y_pred) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_pred - y_true
    return {
        "mae": float(np.mean(np.abs(err))),
        "medae": float(np.median(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "mean_rel_err": float(np.mean(np.abs(err) / np.clip(y_true, 1e-3, None))),
    }


class ResidualDistanceMLP(nn.Module):
    def __init__(self, in_dim: int = 16, hidden=(64, 32)):
        super().__init__()
        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(inplace=True), nn.Dropout(0.1)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class AbsoluteMLP(nn.Module):
    def __init__(self, in_dim: int = 16, hidden=(64, 32)):
        super().__init__()
        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(inplace=True), nn.Dropout(0.1)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


LEDGER: List[Dict[str, Any]] = []


def save_torch_experiment(exp_id: str, approach: str, model: nn.Module, metrics: dict, extra: dict | None = None):
    out = MODELS_DIR / exp_id
    out.mkdir(parents=True, exist_ok=True)
    weights = out / "model.pt"
    torch.save(model.state_dict(), weights)
    # Also refresh the live-inference alias used by Django
    torch.save(model.state_dict(), MODELS_DIR / "distance_nn.pt")
    meta = {
        "experiment_id": exp_id,
        "approach": approach,
        "dataset": DOI,
        "dataset_name": "Human tracking dataset of 3D anatomical landmarks and pose keypoints",
        "feature_names": FEATURE_NAMES,
        "metrics_test": metrics,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "formula": "Z = Z0 * exp(delta)" if "Residual" in approach or "residual" in approach.lower() or "log-Huber" in approach else approach,
        **(extra or {}),
    }
    with (out / "metadata.json").open("w") as f:
        json.dump(meta, f, indent=2)
    LEDGER.append({"experiment_id": exp_id, "approach": approach, **metrics, "weights": str(weights)})
    print(json.dumps(meta, indent=2)[:800])
    return meta


### Experiment 1 — Pinhole \(Z_0\) only (no learning)


In [ ]:
exp_id = "dist_exp01_pinhole_z0"
approach = "Geometry only — pinhole Z0 (no learning)"
pred = z0_te
metrics = regression_metrics(y_te, pred)
print(approach, metrics)
# dummy module so save path stays consistent
class _Const(nn.Module):
    def forward(self, x):
        return x.new_zeros(x.shape[0])
save_torch_experiment(exp_id, approach, _Const(), metrics, extra={"note": "predictions use z0 column, not this module"})


### Experiment 2 — Ridge regression on pose-geometry features


In [ ]:
exp_id = "dist_exp02_ridge"
approach = "Linear baseline — Ridge on pose-geometry features"
ridge = Ridge(alpha=1.0, random_state=SEED)
ridge.fit(X_tr, y_tr)
pred = ridge.predict(X_te)
metrics = regression_metrics(y_te, pred)
print(approach, metrics)
import joblib
(MODELS_DIR / exp_id).mkdir(parents=True, exist_ok=True)
joblib.dump(ridge, MODELS_DIR / exp_id / "ridge.joblib")
LEDGER.append({"experiment_id": exp_id, "approach": approach, **metrics, "weights": str(MODELS_DIR / exp_id / "ridge.joblib")})
with (MODELS_DIR / exp_id / "metadata.json").open("w") as f:
    json.dump({"experiment_id": exp_id, "approach": approach, "metrics_test": metrics, "dataset": DOI}, f, indent=2)


### Experiment 3 — Residual MLP (SAMS production architecture)


In [ ]:
def train_residual(
    X_tr, z0_tr, y_tr, X_te, z0_te, y_te,
    *,
    epochs=40,
    lr=1e-3,
    batch_size=128,
    loss_kind="log_smooth",
):
    model = ResidualDistanceMLP(len(FEATURE_NAMES)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    ds = TensorDataset(
        torch.from_numpy(X_tr), torch.from_numpy(z0_tr), torch.from_numpy(y_tr)
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    best_state, best_mae = None, float("inf")
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for xb, zb, yb in loader:
            xb, zb, yb = xb.to(DEVICE), zb.to(DEVICE), yb.to(DEVICE)
            delta = model(xb).clamp(-1.5, 1.5)
            pred = zb * torch.exp(delta)
            if loss_kind == "log_smooth":
                loss = nn.functional.smooth_l1_loss(torch.log(pred.clamp_min(1e-3)), torch.log(yb.clamp_min(1e-3)))
            elif loss_kind == "log_huber":
                loss = nn.functional.huber_loss(torch.log(pred.clamp_min(1e-3)), torch.log(yb.clamp_min(1e-3)), delta=0.5)
            else:
                loss = nn.functional.smooth_l1_loss(pred, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            total += float(loss.item()) * len(xb)
        model.eval()
        with torch.no_grad():
            xt = torch.from_numpy(X_te).to(DEVICE)
            zt = torch.from_numpy(z0_te).to(DEVICE)
            pred = (zt * torch.exp(model(xt).clamp(-1.5, 1.5))).cpu().numpy()
        m = regression_metrics(y_te, pred)
        history.append({"epoch": epoch, "train_loss": total / len(ds), **m})
        if m["mae"] < best_mae:
            best_mae = m["mae"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if epoch % 10 == 0 or epoch == 1:
            print(f"epoch {epoch:03d}  mae={m['mae']:.3f}  medae={m['medae']:.3f}  rmse={m['rmse']:.3f}")
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = (torch.from_numpy(z0_te).to(DEVICE) * torch.exp(model(torch.from_numpy(X_te).to(DEVICE)).clamp(-1.5, 1.5))).cpu().numpy()
    return model, regression_metrics(y_te, pred), history, pred

exp_id = "dist_exp03_residual_mlp"
approach = "Residual MLP — SAMS Z=Z0*exp(delta)"
model, metrics, hist, pred = train_residual(X_tr, z0_tr, y_tr, X_te, z0_te, y_te, epochs=40)
save_torch_experiment(exp_id, approach, model, metrics, extra={"epochs": 40, "hidden": [64, 32]})
pd.DataFrame(hist).to_csv(MODELS_DIR / exp_id / "history.csv", index=False)


### Experiment 4 — Absolute MLP (no geometric residual)


In [ ]:
def train_absolute(X_tr, y_tr, X_te, y_te, *, epochs=40, lr=1e-3, batch_size=128):
    model = AbsoluteMLP(len(FEATURE_NAMES)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    best_state, best_mae = None, float("inf")
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb).clamp(0.5, 12.0)
            loss = nn.functional.smooth_l1_loss(torch.log(pred), torch.log(yb.clamp_min(1e-3)))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pred = model(torch.from_numpy(X_te).to(DEVICE)).clamp(0.5, 12.0).cpu().numpy()
        m = regression_metrics(y_te, pred)
        if m["mae"] < best_mae:
            best_mae = m["mae"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if epoch % 10 == 0 or epoch == 1:
            print(f"epoch {epoch:03d}  mae={m['mae']:.3f}")
    model.load_state_dict(best_state)
    with torch.no_grad():
        pred = model(torch.from_numpy(X_te).to(DEVICE)).clamp(0.5, 12.0).cpu().numpy()
    return model, regression_metrics(y_te, pred)

exp_id = "dist_exp04_absolute_mlp"
approach = "Absolute MLP — direct Z regression (no Z0 residual)"
model, metrics = train_absolute(X_tr, y_tr, X_te, y_te, epochs=40)
save_torch_experiment(exp_id, approach, model, metrics, extra={"epochs": 40})


### Experiment 5 — Residual MLP on strong-pose rows only


In [ ]:
mask_tr = X_tr[:, FEATURE_NAMES.index("pose_available")] > 0.5
mask_te = X_te[:, FEATURE_NAMES.index("pose_available")] > 0.5
print("pose-available train/test:", mask_tr.sum(), mask_te.sum())

exp_id = "dist_exp05_residual_pose_only"
approach = "Residual MLP — pose_available==1 subset"
model, metrics, hist, pred = train_residual(
    X_tr[mask_tr], z0_tr[mask_tr], y_tr[mask_tr],
    X_te[mask_te], z0_te[mask_te], y_te[mask_te],
    epochs=40,
)
save_torch_experiment(exp_id, approach, model, metrics, extra={"epochs": 40, "filter": "pose_available"})


### Experiment 6 — Residual MLP + Huber on log-distance


In [ ]:
exp_id = "dist_exp06_residual_log_huber"
approach = "Residual MLP — Huber loss on log(Z)"
model, metrics, hist, pred = train_residual(
    X_tr, z0_tr, y_tr, X_te, z0_te, y_te,
    epochs=50, loss_kind="log_huber", lr=8e-4,
)
save_torch_experiment(exp_id, approach, model, metrics, extra={"epochs": 50, "loss": "huber(log Z)"})
pd.DataFrame(hist).to_csv(MODELS_DIR / exp_id / "history.csv", index=False)


## 5. Compare experiments & pick best for live SAMS


In [ ]:
ledger_df = pd.DataFrame(LEDGER).sort_values("mae")
print(ledger_df[["experiment_id", "approach", "mae", "medae", "rmse", "mean_rel_err"]].to_string(index=False))
ledger_path = RESULTS_DIR / "distance_experiment_ledger.csv"
ledger_df.to_csv(ledger_path, index=False)

best = ledger_df.iloc[0]
summary = {
    "task": "vision_camera_person_distance",
    "dataset": DOI,
    "dataset_url": "https://data.mendeley.com/datasets/493s6f753v/2",
    "primary_metric": "mae",
    "best_experiment_id": best["experiment_id"],
    "best_approach": best["approach"],
    "best_metrics": {k: float(best[k]) for k in ["mae", "medae", "rmse", "mean_rel_err"]},
    "models_dir": str(MODELS_DIR),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
with (RESULTS_DIR / "distance_comparison_summary.json").open("w") as f:
    json.dump(summary, f, indent=2)

# Ensure live alias points at best residual-style weights when available
pref = [r for r in LEDGER if "residual" in r["experiment_id"] or "Residual" in r["approach"]]
pick = min(pref, key=lambda r: r["mae"]) if pref else best.to_dict()
best_dir = MODELS_DIR / pick["experiment_id"]
src = best_dir / "model.pt"
if src.exists():
    import shutil
    shutil.copy2(src, MODELS_DIR / "distance_nn.pt")
    print("Live weights ←", src)

# Refresh calib.json next to weights
calib = {
    "fy_px": FY_PX,
    "person_height_m": 1.70,
    "min_dist_m": 0.5,
    "max_dist_m": 12.0,
    "tof_near_m": 1.5,
    "tof_far_m": 2.5,
    "hidden": [64, 32],
    "feature_dim": len(FEATURE_NAMES),
    "feature_names": FEATURE_NAMES,
    "formula": "Z = Z0 * exp(f_theta(x)), Z0 = fy * H / h_px",
    "dataset": DOI,
    "best_experiment_id": pick["experiment_id"],
}
with (MODELS_DIR / "calib.json").open("w") as f:
    json.dump(calib, f, indent=2)

print(json.dumps(summary, indent=2))
print("Ledger:", ledger_path)


In [ ]:
# --- Pred vs GT scatter for best residual run --------------------------------
fig, ax = plt.subplots(figsize=(5.5, 5))
# re-eval exp03-style residual from live weights if present
state_path = MODELS_DIR / "distance_nn.pt"
model = ResidualDistanceMLP(len(FEATURE_NAMES))
model.load_state_dict(torch.load(state_path, map_location="cpu", weights_only=True))
model.eval()
with torch.no_grad():
    pred = (torch.from_numpy(z0_te) * torch.exp(model(torch.from_numpy(X_te)).clamp(-1.5, 1.5))).numpy()
ax.scatter(y_te, pred, s=8, alpha=0.2, c="#2980b9")
lims = [0, max(12, float(y_te.max()), float(pred.max()))]
ax.plot(lims, lims, "k--", lw=1)
ax.set_xlabel("GT distance (m)")
ax.set_ylabel("Predicted (m)")
ax.set_title("Test set — residual MLP")
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
fig_path = RESULTS_DIR / "pred_vs_gt_residual.png"
plt.savefig(fig_path, dpi=140, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)
print(regression_metrics(y_te, pred))


## Next steps (deployment)

1. Copy `models/vision/distance/distance_nn.pt` + `calib.json` into the SAMS repo (already the live paths).
2. Restart Django / `hw.sh run`.
3. Far range → vision NN; near range → set `SAMS_HW_TOF_M` or UART `TOF:<m>`.
4. Optional: enable pose keypoints live with `SAMS_VISION_POSE=1`.
5. Calibrate `fy_px` for your fixed phone mount (stand at 3 m / 5 m with a tape).

Citation for the dataset: Ruescas Nicolau et al., *Human tracking dataset of 3D anatomical landmarks and pose keypoints*, Mendeley Data, V2, doi:10.17632/493s6f753v.2
